# Content-Based Movie Recommendation System

This notebook demonstrates an end-to-end Content-Based Recommendation System using the TMDB 5000 Movies Dataset.
Features like **genres**, **keywords**, and **overview** are extracted, cleaned, and vectorized using Bag-of-Words (`CountVectorizer`).
Similarity between movies is computed using **Cosine Similarity**.

In [ ]:
import ast
import gzip
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Data Loading & Feature Selection

In [ ]:
# Load dataset and select relevant features for content-based filtering
movie = pd.read_csv('tmdb_5000_movies.csv')[['id', 'title', 'genres', 'keywords', 'overview']]
movie.dropna(inplace=True)
movie.reset_index(drop=True, inplace=True)
movie.head()

## 2. Data Cleaning & Feature Extraction

In [ ]:
def convert(obj):
    """Extract names from stringified JSON list and strip spaces."""
    return [i['name'].replace(" ", "") for i in ast.literal_eval(obj)]

movie['genres'] = movie['genres'].apply(convert)
movie['keywords'] = movie['keywords'].apply(convert)
movie['overview'] = movie['overview'].fillna('').astype(str).apply(lambda x: x.split())
movie.head()

## 3. Tag Engineering & Preprocessing

In [ ]:
# Combine extracted features into a single 'tags' string column
df = movie[['id', 'title']].copy()
df['tags'] = (movie['overview'] + movie['genres'] + movie['keywords']).apply(lambda x: " ".join(x).lower())
df.head()

## 4. Vectorization & Cosine Similarity

In [ ]:
# Vectorize text tags using Top 5000 frequent words (excluding English stop words)
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(df['tags']).toarray()

# Compute pairwise cosine similarity matrix
similar = cosine_similarity(vectors)
print(f"Similarity Matrix Shape: {similar.shape}")

## 5. Recommendation Inference Engine

In [ ]:
def recommend(movie_title, top_n=5):
    """Recommends top N similar movies given a target movie title."""
    if movie_title not in df['title'].values:
        print(f"Movie '{movie_title}' not found in dataset.")
        return
    
    movie_index = df[df['title'] == movie_title].index[0]
    distances = similar[movie_index]
    
    # Sort movies based on similarity score (descending order)
    movie_list = sorted(list(enumerate(distances)), key=lambda x: x[1], reverse=True)[1:top_n+1]
    
    print(f"Top recommendations for '{movie_title}':\n")
    for idx, score in movie_list:
        print(f"- {df.iloc[idx].title} (Similarity: {score:.4f})")

# Test recommendations
recommend('Avatar')

## 6. Model Serialization

In [ ]:
# Save compressed artifacts for Streamlit app deployment
with gzip.open('movies.pkl.gz', 'wb') as f:
    pickle.dump(df, f, protocol=pickle.HIGHEST_PROTOCOL)

with gzip.open('similar.pkl.gz', 'wb') as f:
    pickle.dump(similar, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Successfully exported movies.pkl.gz and similar.pkl.gz!")